# Exploratory Data Analysis: Production Memory Safe Results
Visualizing the distributions of ID, Jacobian determinants, and neighbor overlap across models and baselines.

In [ ]:
import pandas as pd
import glob
import os
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
import warnings
warnings.filterwarnings('ignore')

# Helper to load all jsonl files of a given type into a single DataFrame
def load_all_jsonl(file_pattern):
    all_files = glob.glob(f'../results/production_memory_safe/**/{file_pattern}', recursive=True)
    dfs = []
    for file in all_files:
        model = file.split('/')[-3]
        df = pd.read_json(file, lines=True)
        df['model'] = model
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

id_df = load_all_jsonl('id.jsonl')
jacob_df = load_all_jsonl('jacobian.jsonl')
overlap_df = load_all_jsonl('overlap.jsonl')

print(f"Loaded {len(id_df)} ID estimates, {len(jacob_df)} Jacobian blocks, {len(overlap_df)} overlap entries.")

## Intrinsic Dimension Analysis

In [ ]:
# Intrinsic Dimension across depth
if not id_df.empty:
    plt.figure(figsize=(12, 6))
    sns.lineplot(data=id_df[id_df['estimator']=='twonn'], x='depth', y='id_estimate', hue='baseline', style='model', markers=True, dashes=False)
    plt.title('TwoNN Intrinsic Dimension vs Depth')
    plt.ylabel('ID Estimate')
    plt.xlabel('Layer Depth')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

In [ ]:
# Intrinsic Dimension distributions
if not id_df.empty:
    plt.figure(figsize=(12, 6))
    sns.boxplot(data=id_df, x='model', y='id_estimate', hue='baseline')
    plt.title('Distribution of ID Estimates by Model and Baseline')
    plt.show()

## Jacobian Determinant & Condition Numbers

In [ ]:
# Log Abs Determinant across depth
if not jacob_df.empty:
    jacob_df['sub_block_depth_str'] = jacob_df['sub_block_depth'].astype(str)
    plt.figure(figsize=(12, 6))
    sns.lineplot(data=jacob_df, x='sub_block_depth', y='log_abs_det', hue='baseline', style='model', errorbar='sd')
    plt.title('Jacobian Log Abs Det vs Sub-block Depth')
    plt.show()

In [ ]:
# Condition Number distributions (log scale)
if not jacob_df.empty:
    plt.figure(figsize=(12, 6))
    sns.boxplot(data=jacob_df, x='model', y='condition_number', hue='baseline')
    plt.yscale('log')
    plt.title('Condition Number Distribution (Log Scale)')
    plt.show()

## Neighborhood Overlap Analysis

In [ ]:
# Neighborhood Overlap vs Depth To
if not overlap_df.empty:
    plt.figure(figsize=(10, 5))
    sns.lineplot(data=overlap_df, x='depth_to', y='neighbour_overlap', hue='baseline', style='model', markers=True)
    plt.title('Neighborhood Overlap (depth 0 to depth N)')
    plt.xlabel('Target Depth')
    plt.ylabel('Overlap Ratio')
    plt.show()